# 실습3 · ts_rank · ts_rank_cd · 길이 정규화

PostgreSQL 내장 순위 함수를 직접 재 보고, 질문 500개로 검색 품질(Hit@10 · MRR)을 잰다.
강의 : 「PG ts_rank」 · 「PG ts_rank_cd」 · 「PG 길이정규화 옵션」

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 연결과 평가 도구 |
| 3 | ts_rank 와 구역 가중치(setweight) |
| 4 | 같은 단어를 n번 — 등장 횟수에 대한 반응 |
| 5 | ts_rank_cd — 단어가 모여 있을수록 |
| 6 | 길이 정규화 옵션 — 짧은 · 긴 문서 |
| 7 | 정규화 옵션별 상위 10개의 길이 |
| 8 | 문장형 질문 — OR 로 풀고 순위 매기기 |
| 9 | 검색 품질 평가 5종 |
| 10 | 평가 그래프 |

셀 9 는 약 3~4분. 결과는 `results/검색평가.csv` 에 쌓여 실습4·5 의 비교표에 쓰인다.

### 셀 1 · 설정
- `EVAL_N` : 평가에 쓸 질문 수. 줄이면 빨라지지만 실습4·5 와 비교하려면 같은 값으로

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"
EVAL_N = 500                                          # 평가 질문 수 (실습3·4·5 모두 같은 값)
QUESTION = "말라카이트에서 나온 색깔을 사용한 에디션은?"   # 셀 8 의 질문 (정답 지문 id 1)

### 셀 2 · 연결과 평가 도구
- `ORQ` : 문장 → OR tsquery. plainto_tsquery 의 & 를 | 로 바꾼다
- `evaluate(이름, SQL)` : 질문마다 상위 10개 지문 id 를 받아 Hit@10 · MRR 계산

In [ ]:
# 셀 2 · 연결과 평가 도구
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg
from pgvector.psycopg import register_vector

plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False, "figure.dpi": 110})
pd.set_option("display.max_colwidth", 70)

conn = psycopg.connect(DB_URL, autocommit=True)
conn.prepare_threshold = None      # SET 으로 바꾼 설정이 매번 반영되게 (실행 계획 재사용 끔)
register_vector(conn)              # numpy 배열 ↔ vector 타입


def sql(query, params=None):
    """SELECT 결과를 표(DataFrame)로"""
    cur = conn.execute(query, params)
    return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description]) if cur.description else None


# 평가용 질문 : 11개마다 하나씩 고른 EVAL_N 개. 모든 노트북이 같은 질문으로 잰다
E = f"(SELECT id, doc_id AS gold, question, embedding, emb128 FROM questions WHERE id % 11 = 0 ORDER BY id LIMIT {EVAL_N}) e"
RESULTS = Path("results/검색평가.csv")


def evaluate(name, query, params=None):
    """query 는 (질문 id, 정답 지문 id, 상위 문서 id 배열) 을 질문마다 한 행씩 돌려주는 SQL.
    정답 지문이 상위 10위 안이면 Hit, MRR = 정답 순위 역수의 평균. 결과는 RESULTS 에 누적"""
    t = time.perf_counter()
    rows = conn.execute(query, params).fetchall()
    sec = time.perf_counter() - t
    rank = [list(ids[:10]).index(g) + 1 if g in ids[:10] else None for _, g, ids in rows]
    res = {"방법": name, "Hit@10": round(np.mean([r is not None for r in rank]), 3),
           "MRR": round(np.mean([1 / r if r else 0 for r in rank]), 3), "질문수": len(rows), "초": round(sec, 1)}
    RESULTS.parent.mkdir(exist_ok=True)
    new = pd.DataFrame([res])
    if RESULTS.exists():
        old = pd.read_csv(RESULTS)
        new = pd.concat([old[old["방법"] != name], new])
    new.to_csv(RESULTS, index=False)
    return res


ORQ = "replace(plainto_tsquery('korean', {})::text, '&', '|')::tsquery"     # 문장 → OR tsquery
print("평가 질문", conn.execute(f"SELECT count(*) FROM {E}").fetchone()[0], "개")

### 셀 3 · ts_rank 와 구역 가중치
가중치 배열 순서는 **{D, C, B, A}**. 기본값 {0.1, 0.2, 0.4, 1.0} — 제목(A)에 나오면 본문(D)보다 10배.
- `제목에` : 질의어가 제목에 있는 지문
- 왼쪽 : 기본 가중치 · 오른쪽 : 제목(A) 가중치를 본문(D)과 같게 0.1 — 상위 5개가 어떻게 바뀌는지

In [ ]:
# 셀 3 · ts_rank 와 구역 가중치
def top5(weights):
    return sql(f"""SELECT id, title LIKE '%%삼성전자%%' AS 제목에, round(ts_rank('{weights}', tsv, tq)::numeric, 4) AS 점수,
                         left(title, 28) AS 제목
                  FROM docs, plainto_tsquery('korean', '삼성전자') tq WHERE tsv @@ tq
                  ORDER BY ts_rank('{weights}', tsv, tq) DESC, id LIMIT 5""")

pd.concat({"기본 {0.1, 0.2, 0.4, 1.0}": top5("{0.1, 0.2, 0.4, 1.0}"),
           "제목 가중치 끔 {0.1, 0.2, 0.4, 0.1}": top5("{0.1, 0.2, 0.4, 0.1}")}, axis=1)

### 셀 4 · 같은 단어를 n번 — 등장 횟수에 대한 반응
'고양이' 를 1~20번 반복한 문서의 점수. 강의는 ts_rank 가 "등장 횟수에 선형"이라 했다 — 실제 곡선을 본다.

In [ ]:
# 셀 4 · 등장 횟수에 대한 반응
R = sql("""
SELECT n, ts_rank(t, tq) AS ts_rank, ts_rank_cd(t, tq) AS ts_rank_cd
FROM generate_series(1, 20) n,
     LATERAL (SELECT to_tsvector('korean', repeat('고양이 ', n) || '사료 추천') AS t,
                     to_tsquery('korean', '고양이') AS tq) x
""")
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].plot(R["n"], R["ts_rank"], "-o", color="#3D484A"); ax[0].set_title("ts_rank")
ax[1].plot(R["n"], R["ts_rank_cd"], "-o", color="#FF7043"); ax[1].set_title("ts_rank_cd")
for a in ax:
    a.set_xlabel("'고양이' 등장 횟수")
plt.tight_layout(); plt.show()
R.iloc[[0, 1, 2, 4, 9, 19]].round(4)

### 셀 5 · ts_rank_cd — 단어가 모여 있을수록
같은 두 단어, 사이 거리만 다르게. AND(&) 질의와 OR(|) 질의를 비교한다.

In [ ]:
# 셀 5 · ts_rank_cd 밀집도
sql("""
WITH t(문서, body) AS (VALUES
  ('A 붙어 있음', '인공지능 모델 성능 평가 보고서'),
  ('B 8단어 사이', '인공지능 분야 최근 동향 정리 시장 규모 투자 현황 모델 비교'),
  ('C 30단어 사이', '인공지능 ' || repeat('기타 ', 30) || '모델'))
SELECT 문서,
       round(ts_rank(to_tsvector('korean', body), to_tsquery('korean', '인공지능 & 모델'))::numeric, 4)    AS "ts_rank &",
       round(ts_rank_cd(to_tsvector('korean', body), to_tsquery('korean', '인공지능 & 모델'))::numeric, 4) AS "ts_rank_cd &",
       round(ts_rank(to_tsvector('korean', body), to_tsquery('korean', '인공지능 | 모델'))::numeric, 4)    AS "ts_rank |",
       round(ts_rank_cd(to_tsvector('korean', body), to_tsquery('korean', '인공지능 | 모델'))::numeric, 4) AS "ts_rank_cd |"
FROM t
""")

### 셀 6 · 길이 정규화 옵션
세 번째 인자 : 0 없음 · 1 ÷(1 + log 길이) · 2 ÷길이 · 32 ÷(점수 + 1). 같은 질의어를 가진 짧은 · 중간 · 긴 문서로 비교.

In [ ]:
# 셀 6 · 길이 정규화 옵션
sql(f"""
WITH t(문서, body) AS (VALUES
  ('짧음',  '서울 아파트 가격 상승 전망'),
  ('중간',  '서울 아파트 가격 ' || repeat('경제 동향 분석 ', 12) || '상승'),
  ('김',    '서울 아파트 가격 ' || repeat('경제 동향 분석 ', 100) || '상승'))
SELECT 문서, length(to_tsvector('korean', body)) AS 고유형태소, tsv_len(to_tsvector('korean', body)) AS 전체형태소,
       round(ts_rank(to_tsvector('korean', body), tq, 0)::numeric, 4)  AS "0 없음",
       round(ts_rank(to_tsvector('korean', body), tq, 1)::numeric, 4)  AS "1 로그",
       round(ts_rank(to_tsvector('korean', body), tq, 2)::numeric, 4)  AS "2 길이",
       round(ts_rank(to_tsvector('korean', body), tq, 32)::numeric, 4) AS "32 0~1"
FROM t, (SELECT {ORQ.format("'서울 아파트 가격'")} AS tq) x
""")

### 셀 7 · 정규화 옵션별 상위 10개의 길이
실제 지문에서 옵션마다 어떤 길이의 지문이 위로 오는가. 매칭된 전체 평균과 비교한다.

In [ ]:
# 셀 7 · 정규화 옵션별 상위 10개의 평균 길이
sql(f"""
WITH tq AS (SELECT {ORQ.format("'코로나 백신 접종'")} AS tq)
SELECT o.opt AS 옵션,
       (SELECT round(avg(length(content))) FROM (
          SELECT content FROM docs, tq WHERE tsv @@ tq.tq
          ORDER BY ts_rank(tsv, tq.tq, o.opt) DESC, id LIMIT 10) x) AS 상위10_평균글자,
       (SELECT round(avg(length(content))) FROM docs, tq WHERE tsv @@ tq.tq) AS 매칭전체_평균글자
FROM (VALUES (0), (1), (2), (8)) o(opt)
""")

### 셀 8 · 문장형 질문 — OR 로 풀고 순위 매기기
OR 로 풀면 후보가 수천 개. ts_rank 가 정답 지문(★)을 몇 위에 두는가.

In [ ]:
# 셀 8 · 문장형 질문의 순위
sql(f"""
SELECT CASE WHEN id = 1 THEN '★' ELSE '' END AS 정답, id, left(title, 45) AS 제목,
       round(ts_rank(tsv, tq)::numeric, 4) AS ts_rank
FROM docs, (SELECT {ORQ.format('%(q)s')} AS tq) x
WHERE tsv @@ tq
ORDER BY ts_rank(tsv, tq) DESC, id LIMIT 5
""", {"q": QUESTION})

### 셀 9 · 검색 품질 평가 5종
질문 500개 각각에 대해 상위 10개를 뽑아 정답 지문의 순위를 본다. 약 3~4분.
- `AND` : plainto_tsquery 그대로 · `OR ts_rank` · `OR ts_rank_cd` · `OR 정규화1` · `OR 정규화2`

In [ ]:
# 셀 9 · 검색 품질 평가 5종
def ts_query(rank_expr, tq_expr):
    return f"""SELECT e.id, e.gold, ARRAY(SELECT d.id FROM docs d WHERE d.tsv @@ x.tq
                                           ORDER BY {rank_expr} DESC, d.id LIMIT 10)
               FROM {E} CROSS JOIN LATERAL (SELECT {tq_expr} AS tq) x"""

methods = {
    "AND ts_rank":    ts_query("ts_rank(d.tsv, x.tq)", "plainto_tsquery('korean', e.question)"),
    "OR ts_rank":     ts_query("ts_rank(d.tsv, x.tq)", ORQ.format("e.question")),
    "OR ts_rank_cd":  ts_query("ts_rank_cd(d.tsv, x.tq)", ORQ.format("e.question")),
    "OR 정규화1(로그)": ts_query("ts_rank(d.tsv, x.tq, 1)", ORQ.format("e.question")),
    "OR 정규화2(길이)": ts_query("ts_rank(d.tsv, x.tq, 2)", ORQ.format("e.question")),
}
T = pd.DataFrame([evaluate(name, query) for name, query in methods.items()])
T

### 셀 10 · 평가 그래프

In [ ]:
# 셀 10 · 평가 그래프
fig, ax = plt.subplots(figsize=(11, 3.6))
x = np.arange(len(T))
ax.bar(x - 0.2, T["Hit@10"], 0.4, color="#3D484A", label="Hit@10")
ax.bar(x + 0.2, T["MRR"], 0.4, color="#FF7043", label="MRR")
for i, (h, m) in enumerate(T[["Hit@10", "MRR"]].values):
    ax.text(i - 0.2, h, f"{h:.3f}", ha="center", va="bottom"); ax.text(i + 0.2, m, f"{m:.3f}", ha="center", va="bottom")
ax.set_xticks(x, T["방법"]); ax.set_ylim(0, 1); ax.legend(); ax.set_title(f"ts_rank 계열 (질문 {EVAL_N}개)")
plt.show()